Хорошее daily

In [ ]:
import os
import re
import pandas as pd
import numpy as np
from pathlib import Path

# ==========================================
# 1. КОНФИГУРАЦИЯ И ВСПОМОГАТЕЛЬНЫЕ ФУНКЦИИ
# ==========================================

# Названия столбцов для суточных данных с единицами измерения
DAILY_COLS = [
    'День', 'Т воздуха ср, °C', 'Т воздуха макс, °C', 'Т воздуха мин, °C',
    'Т почвы ср (огол), °C', 'Т почвы макс (огол), °C', 'Т почвы мин (огол), °C',
    'Т точки росы мин, °C', 'Парц давл ср, гПа', 'Отн влаж ср, %', 'Отн влаж мин, %',
    'Дефицит ср, гПа', 'Дефицит макс, гПа', 'Атм давл станц, гПа', 'Атм давл море, гПа',
    'Облач общ, баллы', 'Облач ниж, баллы', 'Ветер ср, м/с', 'Ветер макс из 8 сроков, м/с',
    'Ветер абс макс, м/с', 'Осадки за сутки, мм', 'Сост почвы шифр', 'Сост почвы срок',
    'Снег покров шифр', 'Снег покров высота, см'
]

# Названия столбцов для температуры почвы
SOIL_COLS = [
    'День', 'Т 0.05м огол, °C', 'Т 0.10м огол, °C', 'Т 0.15м огол, °C', 'Т 0.20м огол, °C',
    'Т 0.02м ест, °C', 'Т 0.05м ест, °C', 'Т 0.10м ест, °C', 'Т 0.15м ест, °C', 'Т 0.20м ест, °C', 'Т 0.40м ест, °C',
    'Т 0.80м ест, °C', 'Т 1.20м ест, °C', 'Т 1.60м ест, °C', 'Т 2.40м ест, °C', 'Т 3.20м ест, °C',
    'Высота снега, см'
]

def clean_num(val):
    """Очистка от '*' и конвертация в float."""
    if val is None: return np.nan
    val = str(val).replace('*', '').strip()
    if val == '' or val == '-': return np.nan
    try:
        return float(val)
    except ValueError:
        return val

def read_file_safe(filepath):
    """Чтение файла с автоподбором кодировки."""
    for enc in ['utf-8', 'cp1251', 'cp866']:
        try:
            with open(filepath, 'r', encoding=enc) as f:
                return f.read()
        except UnicodeDecodeError:
            continue
    raise ValueError(f"Не удалось определить кодировку для {filepath}")

# ==========================================
# 2. ОСНОВНОЙ ПАРСЕР
# ==========================================

def parse_agg_file(filepath):
    text = read_file_safe(filepath)
    lines = text.split('\n')
    
    # Словари для сбора данных по секциям
    data = {
        'daily': [],
        'phenomena': [],
        'soil': [],
        'monthly_raw': [],
        'precip_raw': []
    }
    
    current_meta = None
    current_section = None
    
    # Переменные для склеивания многострочных явлений
    phen_day = None
    phen_text = []

    def flush_phenomena():
        nonlocal phen_day, phen_text
        if phen_day is not None and current_meta:
            data['phenomena'].append({
                **current_meta,
                'День': phen_day,
                'Описание явлений': ' '.join(phen_text)
            })
        phen_day = None
        phen_text = []

    for line in lines:
        line_stripped = line.strip()
        if not line_stripped:
            continue

        # Проверка на заголовок станции и секции
        # Пример: Станция   Тальменка             N станции  5388360  Год 2017  Месяц   1         С У Т О Ч Н Ы Е   Д А Н Н Ы Е         стр.  7
        meta_match = re.match(r'^Станция\s+(.*?)\s+N\s+(?:станции|Станции)\s+(\d+)\s+Год\s+(\d+)\s+Месяц\s+(\d+)\s+(.*?)\s+стр\.\s+\d+$', line_stripped)
        
        if meta_match:
            flush_phenomena() # Сохраняем накопленные явления перед сменой секции
            
            station_name = meta_match.group(1).strip()
            station_id = meta_match.group(2)
            year = int(meta_match.group(3))
            month = int(meta_match.group(4))
            section_title = meta_match.group(5).strip()
            
            current_meta = {
                'Станция': station_name,
                'ID_Станции': station_id,
                'Год': year,
                'Месяц': month
            }
            
            # Определяем тип секции
            if 'С У Т О Ч Н Ы Е' in section_title:
                current_section = 'daily'
            elif 'А Т М О С Ф Е Р Н Ы Е' in section_title:
                current_section = 'phenomena'
            elif 'М Е С Я Ч Н Ы Е' in section_title:
                current_section = 'monthly'
            elif 'ОСАДКИ c СУММАРНОЙ' in section_title:
                current_section = 'precip'
            elif 'ТЕМПЕРАТУРА ПОЧВЫ' in section_title:
                current_section = 'soil'
            else:
                current_section = 'unknown'
            continue

        if not current_meta or not current_section:
            continue

        # ==============================
        # ПАРСИНГ СУТОЧНЫХ ДАННЫХ
        # ==============================
        if current_section == 'daily':
            # Строки с данными начинаются с числа 1-31
            if re.match(r'^\s*(\d{1,2})\s+', line_stripped):
                parts = line_stripped.split()
                if len(parts) >= 20:
                    row = {**current_meta}
                    for i, col in enumerate(DAILY_COLS):
                        if i < len(parts):
                            val = parts[i]
                            if col == 'День':
                                row[col] = int(val)
                            elif col in ['Сост почвы шифр', 'Снег покров шифр']:
                                row[col] = val
                            else:
                                row[col] = clean_num(val)
                        else:
                            row[col] = np.nan
                    data['daily'].append(row)
                    
            # Сводки (1д, 2д, Мес, Макс, Мин) сохраняем как сырые строки
            elif re.match(r'^(1д|2д|3д|Мес|Максим|Миним)', line_stripped):
                data['monthly_raw'].append({**current_meta, 'Тип_строки': 'Сводка_суток', 'Сырая_строка': line_stripped})

        # ==============================
        # ПАРСИНГ АТМОСФЕРНЫХ ЯВЛЕНИЙ
        # ==============================
        elif current_section == 'phenomena':
            if re.match(r'^\s*(\d{1,2})\s+', line_stripped):
                flush_phenomena()
                phen_day = int(re.match(r'^\s*(\d{1,2})', line_stripped).group(1))
                phen_text = [line_stripped]
            elif phen_day is not None:
                phen_text.append(line_stripped)

        # ==============================
        # ПАРСИНГ ТЕМПЕРАТУРЫ ПОЧВЫ
        # ==============================
        elif current_section == 'soil':
            if re.match(r'^\s*(\d{1,2})\s+', line_stripped) or re.match(r'^(Ср\.|Максим|Миним)', line_stripped):
                parts = line_stripped.split()
                if len(parts) >= 10:
                    row = {**current_meta}
                    for i, col in enumerate(SOIL_COLS):
                        if i < len(parts):
                            val = parts[i]
                            if col == 'День':
                                row[col] = int(val) if val.isdigit() else val
                            else:
                                row[col] = clean_num(val)
                        else:
                            row[col] = np.nan
                    data['soil'].append(row)

        # ==============================
        # ОСАДКИ С ПОПРАВКОЙ И МЕСЯЧНЫЕ
        # ==============================
        elif current_section in ['precip', 'monthly']:
            # Сохраняем все строки с данными как сырые, чтобы не потерять структуру
            if re.match(r'^\s*(\d{1,2})\s+', line_stripped) or re.match(r'^(1д|2д|3д|М\.|Сум\.|Сред|Повт|00|03|06|09|12|15|18|21)', line_stripped):
                data[f'{current_section}_raw'].append({
                    **current_meta, 
                    'Секция': current_section,
                    'Сырая_строка': line_stripped
                })

    flush_phenomena() # Сохраняем последние явления
    return data

# ==========================================
# 3. ОБРАБОТКА ДИРЕКТОРИИ И СОХРАНЕНИЕ
# ==========================================

def process_directory(input_dir, output_excel):
    all_data = {
        'daily': [],
        'phenomena': [],
        'soil': [],
        'monthly_raw': [],
        'precip_raw': []
    }
    
    print(f"🔍 Сканирование директории: {input_dir}")
    for root, _, files in os.walk(input_dir):
        for file in files:
            if file.upper().startswith('AGG') and not file.endswith('.xlsx'):
                filepath = os.path.join(root, file)
                print(f"⚙️ Обработка: {filepath}")
                try:
                    file_data = parse_agg_file(filepath)
                    for key in all_data:
                        all_data[key].extend(file_data[key])
                except Exception as e:
                    print(f"❌ Ошибка при чтении {filepath}: {e}")

    print(f"\n💾 Сохранение данных в {output_excel}...")
    with pd.ExcelWriter(output_excel, engine='openpyxl') as writer:
        
        # 1. Суточные данные
        if all_data['daily']:
            df = pd.DataFrame(all_data['daily'])
            df.to_excel(writer, sheet_name='Суточные_данные', index=False)
            print(f"✅ Суточные данные: {len(df)} строк")
            
        # 2. Атмосферные явления
        if all_data['phenomena']:
            df = pd.DataFrame(all_data['phenomena'])
            df.to_excel(writer, sheet_name='Атм_явления', index=False)
            print(f"✅ Атм. явления: {len(df)} строк")
            
        # 3. Температура почвы
        if all_data['soil']:
            df = pd.DataFrame(all_data['soil'])
            df.to_excel(writer, sheet_name='Температура_почвы', index=False)
            print(f"✅ Температура почвы: {len(df)} строк")
            
        # 4. Месячные выводы (сырые строки)
        if all_data['monthly_raw']:
            df = pd.DataFrame(all_data['monthly_raw'])
            df.to_excel(writer, sheet_name='Месячные_выводы', index=False)
            print(f"✅ Месячные выводы: {len(df)} строк")
            
        # 5. Осадки с поправкой (сырые строки)
        if all_data['precip_raw']:
            df = pd.DataFrame(all_data['precip_raw'])
            df.to_excel(writer, sheet_name='Осадки_с_поправкой', index=False)
            print(f"✅ Осадки с поправкой: {len(df)} строк")

    print(f"\n🎉 Готово! Все данные сохранены в {output_excel}")

if __name__ == "__main__":
    # Укажите вашу директорию с файлами AGG
    INPUT_DIRECTORY = Path(r"E:\Python_evening\Перевод кодировки ТСХ")
    OUTPUT_FILE = "AGG_Parsed_Master_Final.xlsx"
    
    process_directory(INPUT_DIRECTORY, OUTPUT_FILE)

🔍 Сканирование директории: E:\Python_evening\Перевод кодировки ТСХ
⚙️ Обработка: E:\Python_evening\Перевод кодировки ТСХ\New\AGG.117

💾 Сохранение данных в AGG_Parsed_Master_Final.xlsx...
✅ Суточные данные: 930 строк
✅ Атм. явления: 868 строк
✅ Температура почвы: 30 строк
✅ Месячные выводы: 2227 строк

🎉 Готово! Все данные сохранены в AGG_Parsed_Master_Final.xlsx
